In [1]:
first = True

# Detecção de Gestos e Classificação em Libras

In [2]:

import pip


if first:
    !pip install torch 
    !pip install torchvision 
    !pip install pandas
    !pip install av
    !pip install seaborn
    !pip install matplotlib
    !pip install beautifulsoup4
    !pip install opencv-python
    !pip install numpy
    !pip install yt-dlp
    !pip install tqdm
    !pip install mediapipe
    !pip install kagglehub
    !pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

    
first = False

import torchvision.models as models
import torch.optim as optim
import kagglehub
import pandas as pd 
import seaborn as sns 
import numpy as np 
import matplotlib.pyplot as plt 
import torch 
from torch import nn
import cv2 
from PIL import Image
from bs4 import BeautifulSoup
import requests
import yt_dlp
from torchvision import models
import mediapipe as mp
import os 
import tqdm 
from tqdm import tqdm
import av

Looking in indexes: https://download.pytorch.org/whl/cu118
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
     ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
     ----- ---------------------------------- 0.5/4.0 MB 5.7 MB/s eta 0:00:01
     ---------------------------------------- 4.0/4.0 MB 13.4 MB/s eta 0:00:00
     ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
     ---------------------------------------- 4.0/4.0 MB 48.3 MB/s eta 0:00:00
     ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
     ---------------------------------------- 4.0/4.0 MB 21.9 MB/s eta 0:00:00
     ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
     ---------------------------------------- 4.0/4.0 MB 39.9 MB/s eta 0:00:00
     ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
     ----------------------------------------

c:\Users\anton\miniforge3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Download das Ferramentas
Temos que por se tratar de um algoritmo de aprendizado de máquina que vai ser usado em sistemas reais, temos que 
primeiramente, executar seu treinamento de forma que ele possa ser usado. Para isso, temos que o **V-Librasil** é um 
dataset público de libras que pode ser usado para criar sistemas capazes de entender a libras, e que pode ser usado para o treinamento 
de sistemas de aprendizado de máquina para essa linguagem

In [3]:
INES_DICT = "https://dicionario.ines.gov.br/"

# ─── Diagnóstico CUDA ──────────────────────────────────────────────────────────
print("="*60)
print("DIAGNÓSTICO CUDA/GPU")
print("="*60)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ CUDA não está disponível!")
    print("Para usar GPU, reinstale PyTorch com suporte CUDA:")
    print("  pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")
print("="*60 + "\n")

# ─── Dataset ──────────────────────────────────────────────────────────────────
print("Downloading vlibrasil dataset")
path = kagglehub.dataset_download("davimedio01/v-librasil")
print("Path to dataset files:", path)
 
videos_dir_path = path + "/videos UFPE (V-LIBRASIL)"
print("Path to videos directory:", videos_dir_path)
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
 # ─── Carrega caminhos e labels ─────────────────────────────────────────────────
video_paths = []
label_list = []
 
class_names = sorted(os.listdir(videos_dir_path))
 
for i, class_name in enumerate(class_names):
    class_dir = os.path.join(videos_dir_path, class_name)
    if os.path.isdir(class_dir):
        for video_file in os.listdir(class_dir):
            if video_file.endswith(('.mp4', '.avi', '.mov')):
                video_paths.append(os.path.join(class_dir, video_file))
                label_list.append(i)
 
num_classes = len(class_names)
print(f"Found {len(video_paths)} videos, {num_classes} classes")
 
 
# ─── Função auxiliar: lê vídeo com PyAV (suporta interlaced) ──────────────────
def read_video_pyav(video_path, num_frames):
    """
    Lê frames de um vídeo usando PyAV.
    Suporta vídeos entrelaçados (interlaced) que o OpenCV não consegue ler.
    Retorna lista de arrays numpy (H, W, 3) em RGB.
    """
    frames = []
    try:
        container = av.open(video_path)
 
        for frame in container.decode(video=0):
            # to_ndarray com format="rgb24" faz o desentrelaçamento automaticamente
            frame_rgb = frame.to_ndarray(format="rgb24")
 
            frame_resized = cv2.resize(frame_rgb, (224, 224))
            frames.append(frame_resized)
 
            if len(frames) >= num_frames:
                break
 
        container.close()
 
    except Exception as e:
        print(f"Erro ao abrir {video_path}: {e}")
 
    # Padding: repete o último frame se o vídeo for mais curto que num_frames
    if len(frames) == 0:
        frames = [np.zeros((224, 224, 3), dtype=np.uint8)]
 
    while len(frames) < num_frames:
        frames.append(frames[-1])
 
    return frames[:num_frames]


DIAGNÓSTICO CUDA/GPU
PyTorch version: 2.7.1+cu118
CUDA available: False
⚠️ CUDA não está disponível!
Para usar GPU, reinstale PyTorch com suporte CUDA:
  pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118



c:\Users\anton\miniforge3\Lib\site-packages\torch\cuda\__init__.py:174: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 1: invalid argument (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


Path to dataset files: C:\Users\anton\.cache\kagglehub\datasets\davimedio01\v-librasil\versions\1
Path to videos directory: C:\Users\anton\.cache\kagglehub\datasets\davimedio01\v-librasil\versions\1/videos UFPE (V-LIBRASIL)
Using device: cpu
Found 4086 videos, 3 classes


In [4]:
# ─── Feature cache / Dataset simplificado ─────────────────────────────────────
FEATURES_CACHE_PATH = "features_cache.pt"

def extract_and_cache_features(video_paths, labels, num_frames, device, batch_size=16):
    if os.path.exists(FEATURES_CACHE_PATH):
        print("✅ Carregando features do cache...")
        data = torch.load(FEATURES_CACHE_PATH)
        return data["features"], data["labels"]

    print("⚙️ Extraindo features (uma vez)...")
    feature_extractor = models.resnet18(weights=None)
    state_dict = torch.load("models/resnet18-f37072fd.pth", map_location=device)
    feature_extractor.load_state_dict(state_dict)
    feature_extractor.fc = nn.Identity()
    feature_extractor = feature_extractor.to(device)
    feature_extractor.eval()

    all_features = []
    all_labels = []

    with torch.no_grad():
        for idx in tqdm(range(len(video_paths)), desc="Extraindo features"):
            frames = read_video_pyav(video_paths[idx], num_frames)

            video_tensor = torch.stack([
                torch.from_numpy(f).permute(2, 0, 1) for f in frames
            ]).float() / 255.0

            video_tensor = video_tensor.to(device)
            features = feature_extractor(video_tensor)  # [T, 512]

            all_features.append(features.cpu())
            all_labels.append(labels[idx])

    all_features = torch.stack(all_features)  # [N, T, 512]
    all_labels = torch.tensor(all_labels, dtype=torch.long)

    torch.save({"features": all_features, "labels": all_labels}, FEATURES_CACHE_PATH)
    print(f"✅ Features salvas em {FEATURES_CACHE_PATH}")

    return all_features, all_labels


class FeaturesDataset(torch.utils.data.Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


In [5]:
# ─── Modelo ────────────────────────────────────────────────────────────────────
class LibrasClassifier(nn.Module):
    def __init__(self, hidden_size, num_layers, num_classes):
        super(LibrasClassifier, self).__init__()

        # A extração de features já é feita antes do treinamento.
        self.lstm = nn.LSTM(512, hidden_size, num_layers, batch_first=True)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x: [batch, T, 512]
        out, (hn, cn) = self.lstm(x)
        out = self.classifier(out[:, -1, :])
        return out


In [8]:
# ─── Extração de features e DataLoader ─────────────────────────────────────────
NUM_FRAMES = 16  # Ajuste conforme seus vídeos

features, labels_tensor = extract_and_cache_features(
    video_paths,
    label_list,
    num_frames=NUM_FRAMES,
    device=device,
    batch_size=16,
)

dataset = FeaturesDataset(features, labels_tensor)

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,        # Windows: usar 0 para evitar multiprocessing issues
    pin_memory=True,
)


✅ Carregando features do cache...


In [9]:
# ─── Treinamento ───────────────────────────────────────────────────────────────

model = LibrasClassifier(
    hidden_size=256,
    num_layers=2,
    num_classes=num_classes
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    loop = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=True)

    for features_batch, labels_batch in loop:
        features_batch = features_batch.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()

        outputs = model(features_batch)
        loss = criterion(outputs, labels_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Atualiza a loss em tempo real na barra
        loop.set_postfix(loss=loss.item())

    avg_loss = running_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs} finalizada — Loss média: {avg_loss:.4f}\n")


Epoch 1/10: 100%|██████████| 64/64 [00:03<00:00, 21.25it/s, loss=1.34e-5] 


Epoch 1/10 finalizada — Loss média: 0.0336



Epoch 2/10: 100%|██████████| 64/64 [00:02<00:00, 22.11it/s, loss=1.2e-5] 


Epoch 2/10 finalizada — Loss média: 0.0000



Epoch 3/10: 100%|██████████| 64/64 [00:02<00:00, 21.49it/s, loss=1.08e-5]


...

Epoch 8/10 finalizada — Loss média: 0.0000



Epoch 9/10: 100%|██████████| 64/64 [00:02<00:00, 23.08it/s, loss=5.72e-6]


Epoch 9/10 finalizada — Loss média: 0.0000



Epoch 10/10: 100%|██████████| 64/64 [00:02<00:00, 24.11it/s, loss=5.13e-6]

Epoch 10/10 finalizada — Loss média: 0.0000



In [14]:
# ─── Reconhecimento em Tempo Real ──────────────────────────────────────────────

# Salva o modelo treinado
torch.save(model.state_dict(), "libras_classifier.pth")
print("✅ Modelo salvo em libras_classifier.pth")

# ─── Carrega o modelo para inferência ───────────────────────────────────────────
def load_model_for_inference(model_path, device):
    """Carrega o modelo treinado para inferência"""
    model = LibrasClassifier(
        hidden_size=256,
        num_layers=2,
        num_classes=num_classes
    ).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    return model


# ─── Extrator de features para inferência ───────────────────────────────────────
def get_feature_extractor(device):
    """Cria o ResNet18 para extrair features"""
    feature_extractor = models.resnet18(weights=None)
    state_dict = torch.load("models/resnet18-f37072fd.pth", map_location=device)
    feature_extractor.load_state_dict(state_dict)
    feature_extractor.fc = nn.Identity()
    feature_extractor = feature_extractor.to(device)
    feature_extractor.eval()
    return feature_extractor


# ─── Inferência em Tempo Real com Feedback Visual ───────────────────────────────
def recognize_gestures_webcam(model, feature_extractor, class_names, device, num_frames=16, confidence_threshold=0.5):
    """
    Captura vídeo da webcam e reconhece gestos em tempo real com feedback visual.
    Pressione 'q' para sair.
    
    Args:
        model: Modelo LSTM treinado
        feature_extractor: ResNet18 para extrair features
        class_names: Lista de nomes dos gestos
        device: Dispositivo (cuda/cpu)
        num_frames: Número de frames necessários para predição
        confidence_threshold: Mínimo de confiança para considerar predição válida
    """
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("❌ Não foi possível acessar a webcam")
        return
    
    print("✅ Webcam aberta. Pressione 'q' para sair.")
    print(f"📌 Precisa de {num_frames} frames para fazer predição")
    
    frame_buffer = []
    prediction_history = []  # Histórico das últimas predições
    max_history = 10
    
    with torch.no_grad():
        while True:
            ret, frame = cap.read()
            
            if not ret:
                print("❌ Erro ao capturar frame")
                break
            
            # Inverte a imagem horizontalmente (espelho)
            frame = cv2.flip(frame, 1)
            display_frame = frame.copy()
            
            # Redimensiona para 224x224 (mesmo tamanho do treinamento)
            frame_resized = cv2.resize(frame, (224, 224))
            frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
            
            # Adiciona frame ao buffer
            frame_buffer.append(frame_rgb)
            
            # Mantém apenas os últimos num_frames frames
            if len(frame_buffer) > num_frames:
                frame_buffer.pop(0)
            
            height, width = display_frame.shape[:2]
            
            # ───── PAINEL ESQUERDO: STATUS E INFORMAÇÕES ─────────────────────────
            # Background semi-transparente
            overlay = display_frame.copy()
            cv2.rectangle(overlay, (10, 10), (400, 300), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.3, display_frame, 0.7, 0, display_frame)
            
            # Status do buffer
            buffer_progress = len(frame_buffer) / num_frames
            status_color = (0, 255, 0) if len(frame_buffer) == num_frames else (0, 165, 255)
            
            cv2.putText(display_frame, "STATUS", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 
                       1, (255, 255, 255), 2)
            cv2.putText(display_frame, f"Frames: {len(frame_buffer)}/{num_frames}", (20, 70), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)
            
            # Barra de progresso do buffer
            bar_width = 350
            bar_height = 20
            filled_width = int(bar_width * buffer_progress)
            cv2.rectangle(display_frame, (20, 90), (370, 110), (100, 100, 100), 2)
            if filled_width > 0:
                cv2.rectangle(display_frame, (20, 90), (20 + filled_width, 110), status_color, -1)
            
            # Mensagem de status
            if len(frame_buffer) < num_frames:
                msg = f"Aguarde... {len(frame_buffer)}/{num_frames}"
                cv2.putText(display_frame, msg, (20, 150), cv2.FONT_HERSHEY_SIMPLEX, 
                           0.7, (100, 100, 255), 2)
            else:
                cv2.putText(display_frame, "PRONTO! (Predição em andamento)", (20, 150), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            
            # ───── PREDIÇÃO ──────────────────────────────────────────────────────
            if len(frame_buffer) == num_frames:
                # Converte para tensor
                video_tensor = torch.stack([
                    torch.from_numpy(f).permute(2, 0, 1) for f in frame_buffer
                ]).float() / 255.0
                
                video_tensor = video_tensor.to(device)
                
                # Extrai features
                features = feature_extractor(video_tensor)  # [T, 512]
                features = features.unsqueeze(0)  # [1, T, 512]
                
                # Predição
                outputs = model(features)
                probabilities = torch.softmax(outputs, dim=1)[0]
                
                # Top 3 predições
                top_3_probs, top_3_indices = torch.topk(probabilities, min(3, len(class_names)))
                
                predicted_class = top_3_indices[0].item()
                confidence = top_3_probs[0].item()
                gesture_name = class_names[predicted_class]
                
                # Adiciona ao histórico
                prediction_history.append((gesture_name, confidence))
                if len(prediction_history) > max_history:
                    prediction_history.pop(0)
                
                # ───── PAINEL SUPERIOR: PREDIÇÃO PRINCIPAL ────────────────────────
                # Background para predição
                overlay = display_frame.copy()
                cv2.rectangle(overlay, (width - 400, 10), (width - 10, 150), (0, 0, 0), -1)
                cv2.addWeighted(overlay, 0.3, display_frame, 0.7, 0, display_frame)
                
                # Cor baseada em confiança
                if confidence >= 0.8:
                    pred_color = (0, 255, 0)  # Verde
                    confidence_text = "🟢 MUITO CONFIANTE"
                elif confidence >= 0.6:
                    pred_color = (0, 255, 255)  # Amarelo
                    confidence_text = "🟡 CONFIANTE"
                else:
                    pred_color = (0, 165, 255)  # Laranja
                    confidence_text = "🟠 BAIXA CONFIANÇA"
                
                cv2.putText(display_frame, "PREDIÇÃO", (width - 390, 40), 
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
                cv2.putText(display_frame, gesture_name, (width - 390, 80), 
                           cv2.FONT_HERSHEY_SIMPLEX, 1.2, pred_color, 3)
                cv2.putText(display_frame, f"{confidence:.1%}", (width - 390, 120), 
                           cv2.FONT_HERSHEY_SIMPLEX, 1, pred_color, 2)
                
                # ───── PAINEL DIREITO: TOP 3 CANDIDATOS ──────────────────────────
                overlay = display_frame.copy()
                cv2.rectangle(overlay, (width - 400, 160), (width - 10, 300), (0, 0, 0), -1)
                cv2.addWeighted(overlay, 0.3, display_frame, 0.7, 0, display_frame)
                
                cv2.putText(display_frame, "TOP 3", (width - 390, 185), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
                
                for i, (prob, idx) in enumerate(zip(top_3_probs, top_3_indices)):
                    y_pos = 220 + (i * 25)
                    class_name = class_names[idx.item()]
                    prob_val = prob.item()
                    
                    # Barra horizontal de probabilidade
                    bar_width = 120
                    filled = int(bar_width * prob_val)
                    cv2.rectangle(display_frame, (width - 240, y_pos - 10), 
                                (width - 120, y_pos + 5), (50, 50, 50), 1)
                    cv2.rectangle(display_frame, (width - 240, y_pos - 10), 
                                (width - 240 + filled, y_pos + 5), (0, 255, 0), -1)
                    
                    cv2.putText(display_frame, f"{i+1}. {class_name}: {prob_val:.0%}", 
                               (width - 380, y_pos + 3), cv2.FONT_HERSHEY_SIMPLEX, 0.6, 
                               (255, 255, 255), 1)
            
            # ───── HISTÓRICO DE PREDIÇÕES (parte inferior) ────────────────────────
            if prediction_history:
                overlay = display_frame.copy()
                cv2.rectangle(overlay, (10, height - 150), (400, height - 10), (0, 0, 0), -1)
                cv2.addWeighted(overlay, 0.3, display_frame, 0.7, 0, display_frame)
                
                cv2.putText(display_frame, "HISTORICO", (20, height - 120), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
                
                for i, (gesture, conf) in enumerate(prediction_history[-5:]):
                    y_pos = height - 85 + (i * 20)
                    color = (0, 255, 0) if conf >= 0.8 else (0, 165, 255)
                    cv2.putText(display_frame, f"• {gesture} ({conf:.0%})", 
                               (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
            
            # ───── INSTRUÇÕES ────────────────────────────────────────────────────
            cv2.putText(display_frame, "Pressione 'q' para sair", (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
            
            # Mostra a imagem
            cv2.imshow("Libras Gesture Recognition - Com Feedback", display_frame)
            
            # Sai se pressionar 'q'
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    
    cap.release()
    cv2.destroyAllWindows()
    print("✅ Webcam fechada")
    print(f"📊 Total de predições: {len(prediction_history)}")


# Carrega o modelo
inference_model = load_model_for_inference("libras_classifier.pth", device)
feature_extractor = get_feature_extractor(device)

# Uncomment a linha abaixo para iniciar o reconhecimento em tempo real com feedback
recognize_gestures_webcam(inference_model, feature_extractor, class_names, device, NUM_FRAMES)


✅ Modelo salvo em libras_classifier.pth
✅ Webcam aberta. Pressione 'q' para sair.
📌 Precisa de 16 frames para fazer predição
✅ Webcam fechada
📊 Total de predições: 10


In [ ]:
# ─── Validação e Métricas ──────────────────────────────────────────────────────

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

# ─── Divide dataset em train/val/test ───────────────────────────────────────────
indices = np.arange(len(features))
train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42, stratify=labels_tensor.numpy())
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42, stratify=labels_tensor[temp_idx].numpy())

train_features = features[train_idx]
train_labels = labels_tensor[train_idx]

val_features = features[val_idx]
val_labels = labels_tensor[val_idx]

test_features = features[test_idx]
test_labels = labels_tensor[test_idx]

print(f"Train: {len(train_features)} | Val: {len(val_features)} | Test: {len(test_features)}")

# ─── Avaliação no conjunto de teste ────────────────────────────────────────────
def evaluate_model(model, test_features, test_labels, device):
    """Avalia o modelo no conjunto de teste e retorna métricas"""
    model.eval()
    all_preds = []
    all_true = []
    
    with torch.no_grad():
        for i in range(len(test_features)):
            feat = test_features[i].unsqueeze(0).to(device)  # [1, T, 512]
            true_label = test_labels[i].item()
            
            output = model(feat)
            pred = torch.argmax(output, dim=1).item()
            
            all_preds.append(pred)
            all_true.append(true_label)
    
    all_preds = np.array(all_preds)
    all_true = np.array(all_true)
    
    # Calcula métricas
    accuracy = accuracy_score(all_true, all_preds)
    precision = precision_score(all_true, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_true, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_true, all_preds, average='weighted', zero_division=0)
    
    return accuracy, precision, recall, f1, all_preds, all_true


# Avalia o modelo
accuracy, precision, recall, f1, preds, true_labels = evaluate_model(
    inference_model, test_features, test_labels, device
)

print("\n" + "="*60)
print("RESULTADOS DA AVALIAÇÃO")
print("="*60)
print(f"Acurácia:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precisão:  {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("="*60 + "\n")

# ─── Relatório de classificação por classe ──────────────────────────────────────
print("Relatório por classe:")
print(classification_report(true_labels, preds, target_names=class_names, zero_division=0))

# ─── Matriz de confusão ─────────────────────────────────────────────────────────
cm = confusion_matrix(true_labels, preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title("Matriz de Confusão")
plt.ylabel("Verdadeiro")
plt.xlabel("Predito")
plt.tight_layout()
plt.show()

# ─── Teste em um vídeo individual ───────────────────────────────────────────────
def test_on_single_video(video_path, model, feature_extractor, class_names, num_frames, device):
    """Testa o modelo em um vídeo específico"""
    frames = read_video_pyav(video_path, num_frames)
    
    video_tensor = torch.stack([
        torch.from_numpy(f).permute(2, 0, 1) for f in frames
    ]).float() / 255.0
    
    video_tensor = video_tensor.to(device)
    
    with torch.no_grad():
        features = feature_extractor(video_tensor)  # [T, 512]
        features = features.unsqueeze(0)  # [1, T, 512]
        
        output = model(features)
        probabilities = torch.softmax(output, dim=1)[0]
    
    # Mostra top 3 predições
    top_3_probs, top_3_indices = torch.topk(probabilities, 3)
    
    print(f"\n📹 Teste de vídeo: {video_path}")
    print("-" * 50)
    for i, (idx, prob) in enumerate(zip(top_3_indices, top_3_probs)):
        print(f"{i+1}. {class_names[idx.item()]}: {prob.item():.2%}")


# Testa em alguns vídeos aleatórios do conjunto de teste
test_video_indices = np.random.choice(test_idx, size=min(3, len(test_idx)), replace=False)

print("\n" + "="*60)
print("TESTES EM VÍDEOS INDIVIDUAIS")
print("="*60)

for i, idx in enumerate(test_video_indices):
    true_label = class_names[labels_tensor[idx].item()]
    print(f"\n✓ Vídeo {i+1} (Verdadeiro: {true_label})")
    test_on_single_video(video_paths[idx], inference_model, feature_extractor, 
                         class_names, NUM_FRAMES, device)
